In [ ]:
import pandas as pd
import os
from collections import Counter

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')

In [ ]:
result_df = pd.read_csv("../data/prediction_comparison_shap_tokens_removed.csv")

In [ ]:
# Female → Male flips (true_label = 0, debiased model incorrectly predicted 1)
flip_f_to_m = result_df[result_df["true_label"] == 0]

# Male → Female flips (true_label = 1, debiased model incorrectly predicted 0)
flip_m_to_f = result_df[result_df["true_label"] == 1]


In [ ]:
def get_token_counts(df):
  tokens = []
  for toks in df["removed_tokens"]:
      if pd.isna(toks) or toks.strip() == "":
          continue  # skip if NaN or empty string
      tokens.extend([token.strip().lower() for token in toks.split(",") if token.strip()])
  return Counter(tokens)

# Count removed tokens by flip direction
f_to_m_counts = get_token_counts(flip_f_to_m)
m_to_f_counts = get_token_counts(flip_m_to_f)

In [ ]:
print("Top tokens whose removal flipped Female → Male:")
print(f_to_m_counts.most_common(10))

print("\nTop tokens whose removal flipped Male → Female:")
print(m_to_f_counts.most_common(10))


Top tokens whose removal flipped Female → Male:
[('impressed', 14), ('liked', 6), ('bedside', 6), ('thoughtful', 4), ('stellar', 3), ('women', 2), ('united', 2), ('humble', 2), ('distinguished', 2), ('acquainted', 2)]

Top tokens whose removal flipped Male → Female:
[('impressed', 23), ('humble', 8), ('summary', 8), ('reliable', 7), ('bedside', 7), ('thoughtful', 5), ('remained', 4), ('respectful', 3), ('distinguished', 3), ('infectious', 2)]


In [ ]:
# Get all unique tokens across both directions
all_tokens = set(f_to_m_counts.keys()).union(set(m_to_f_counts.keys()))

# Compute absolute difference for each token
token_diffs = {
    token: abs(f_to_m_counts.get(token, 0) - m_to_f_counts.get(token, 0))
    for token in all_tokens
}

# Sort tokens by greatest difference
diff_counter = Counter(token_diffs)
top_diff_tokens = diff_counter.most_common(60)  # Top 20 by absolute difference

# Convert to DataFrame for display
diff_df = pd.DataFrame(top_diff_tokens, columns=["token", "abs_diff"])
diff_df["f_to_m_count"] = diff_df["token"].apply(lambda t: f_to_m_counts.get(t, 0))
diff_df["m_to_f_count"] = diff_df["token"].apply(lambda t: m_to_f_counts.get(t, 0))

# Reorder columns
diff_df = diff_df[["token", "f_to_m_count", "m_to_f_count", "abs_diff"]]


In [ ]:
diff_df

,token,f_to_m_count,m_to_f_count,abs_diff
0,impressed,14,23,9
1,summary,1,8,7
2,humble,2,8,6
3,reliable,1,7,6
4,liked,6,1,5
5,remained,1,4,3
6,polite,0,2,2
7,humanitarian,0,2,2
8,stellar,3,1,2
9,acquainted,2,0,2


# Normalize by Total Number of Female and Male Letters

In [ ]:
N_female = (result_df["true_label"] == 0).sum()
N_male = (result_df["true_label"] == 1).sum()

diff_df["normalized_f_to_m"] = diff_df["f_to_m_count"] / N_female
diff_df["normalized_m_to_f"] = diff_df["m_to_f_count"] / N_male

# Optional: Add absolute difference in flip rates
diff_df["normalized_diff"] = abs(diff_df["normalized_f_to_m"] - diff_df["normalized_m_to_f"])
diff_df = diff_df.sort_values(by='normalized_diff', ascending=False)

In [ ]:
diff_df

,token,f_to_m_count,m_to_f_count,abs_diff,normalized_f_to_m,normalized_m_to_f,normalized_diff
4,liked,6,1,5,0.157895,0.010309,0.147585
0,impressed,14,23,9,0.368421,0.237113,0.131308
24,bedside,6,7,1,0.157895,0.072165,0.085730
8,stellar,3,1,2,0.078947,0.010309,0.068638
1,summary,1,8,7,0.026316,0.082474,0.056158
22,thoughtful,4,5,1,0.105263,0.051546,0.053717
9,acquainted,2,0,2,0.052632,0.000000,0.052632
10,women,2,0,2,0.052632,0.000000,0.052632
3,reliable,1,7,6,0.026316,0.072165,0.045849
14,united,2,1,1,0.052632,0.010309,0.042322


In [ ]:
# diff_df.to_csv('../data/shap_token_flip_differences_normalized.csv', index=False)

# Generate Subsamples of Majority Class and Average Results

In [ ]:
# Split the result_df into female and male
df_female = result_df[result_df["true_label"] == 0]
df_male = result_df[result_df["true_label"] == 1]

# Number of subsampling runs and size of minority class
n_runs = 1000
minority_size = min(len(df_female), len(df_male))

# Accumulate token counts across runs
f_to_m_sum = Counter()
m_to_f_sum = Counter()

def count_tokens(df):
    tokens = []
    for toks in df["removed_tokens"]:
        if pd.isna(toks) or toks.strip() == "":
            continue
        tokens.extend([token.strip().lower() for token in toks.split(",") if token.strip()])
    return Counter(tokens)

for _ in range(n_runs):
    # Randomly subsample from each group
    sample_f = df_female.sample(minority_size, replace=False, random_state=None)
    sample_m = df_male.sample(minority_size, replace=False, random_state=None)

    f_to_m_sum.update(count_tokens(sample_f))
    m_to_f_sum.update(count_tokens(sample_m))

# Average the counts over runs
f_to_m_avg = {token: round(count / n_runs, 2) for token, count in f_to_m_sum.items()}
m_to_f_avg = {token: round(count / n_runs, 2) for token, count in m_to_f_sum.items()}

# Combine into final DataFrame
all_tokens = set(f_to_m_avg.keys()).union(m_to_f_avg.keys())
data = []

for token in all_tokens:
    f = f_to_m_avg.get(token, 0)
    m = m_to_f_avg.get(token, 0)
    data.append({
        "token": token,
        "f_to_m_count": f,
        "m_to_f_count": m,
        "abs_diff": round(abs(f - m), 2)
    })

diff_df_subsample_approach = pd.DataFrame(data).sort_values(by="abs_diff", ascending=False).reset_index(drop=True)

In [ ]:
diff_df_subsample_approach

,token,f_to_m_count,m_to_f_count,abs_diff
0,liked,6.0,0.38,5.62
1,impressed,14.0,8.93,5.07
2,bedside,6.0,2.80,3.20
3,stellar,3.0,0.37,2.63
4,summary,1.0,3.16,2.16
5,thoughtful,4.0,1.97,2.03
6,acquainted,2.0,0.00,2.00
7,women,2.0,0.00,2.00
8,reliable,1.0,2.78,1.78
9,united,2.0,0.38,1.62


In [ ]:
# diff_df_subsample_approach.to_csv('../data/shap_token_flip_differences_subsampled.csv', index=False)

In [ ]:
# diff_df.to_csv("../data/tfidf_token_flip_differences.csv", index=False)